# Predictive Modeling Using Machine Learning

## Overview
This notebook demonstrates a complete machine learning workflow:
- Data loading and exploratory analysis
- Data preprocessing and feature engineering
- Model training (Linear/Logistic Regression, Decision Tree, Random Forest)
- Hyperparameter tuning
- Comprehensive evaluation and visualization
- Model persistence

**Goal**: Build accurate predictive models for classification and regression tasks

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
from pathlib import Path

# ML libraries
from sklearn.datasets import load_iris, make_classification, make_regression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, f_regression

# Models
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc, roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score
)

# Settings
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

print("✓ All libraries imported successfully!")

## 2. Load Dataset

In [ ]:
# Load Iris dataset (classification task)
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='species')

# Display dataset information
print("Dataset Shape:", X.shape)
print("\nFirst few rows:")
print(X.head())
print("\nColumn Data Types:")
print(X.dtypes)
print("\nBasic Statistics:")
print(X.describe())
print("\nTarget variable distribution:")
print(y.value_counts().sort_index())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Statistical summary
print("Summary Statistics:")
print(X.describe())

# Correlation matrix
fig, ax = plt.subplots(figsize=(10, 8))
correlation_matrix = X.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Distribution plots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for idx, col in enumerate(X.columns):
    ax = axes[idx // 2, idx % 2]
    ax.hist(X[col], bins=20, edgecolor='black', alpha=0.7)
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.set_title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

# Box plots by target class
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for idx, col in enumerate(X.columns):
    ax = axes[idx // 2, idx % 2]
    df_plot = pd.DataFrame({'value': X[col], 'species': y})
    sns.boxplot(data=df_plot, x='species', y='value', ax=ax)
    ax.set_ylabel(col)
    ax.set_title(f'{col} by Species')
plt.tight_layout()
plt.show()

print("\n✓ EDA completed!")

## 4. Data Cleaning & Preprocessing

In [ ]:
# Check for missing values
print("Missing values:")
print(X.isnull().sum())
print("\nDataset has no missing values - great!")

# Check for duplicates
print(f"\nDuplicate rows: {X.duplicated().sum()}")

# Initialize scaler
scaler = StandardScaler()

# Fit and transform the data (we'll split first, then scale)
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("\nAfter scaling:")
print("Mean:", X_scaled.mean().round(4))
print("Std:", X_scaled.std().round(4))

print("\n✓ Preprocessing completed!")

## 5. Feature Engineering & Selection

In [ ]:
# Feature selection using SelectKBest
selector = SelectKBest(score_func=f_classif, k=3)
X_selected = selector.fit_transform(X_scaled, y)

# Get selected feature names
selected_features = X.columns[selector.get_support()].tolist()
print("Selected features (top 3):")
print(selected_features)

# Feature scores
feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'Score': selector.scores_
}).sort_values('Score', ascending=False)

print("\nFeature Importance Scores:")
print(feature_scores)

# Visualize feature scores
plt.figure(figsize=(10, 6))
plt.barh(feature_scores['Feature'], feature_scores['Score'])
plt.xlabel('Score')
plt.title('Feature Importance Scores')
plt.tight_layout()
plt.show()

# For now, use all features
X_processed = X_scaled

print("\n✓ Feature engineering completed!")

## 6. Train/Validation Split & Cross-Validation

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTarget distribution in training set:")
print(y_train.value_counts().sort_index())
print(f"\nTarget distribution in test set:")
print(y_test.value_counts().sort_index())

# K-fold cross-validation setup
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

print("\n✓ Train-test split completed!")

## 7. Baseline Models (Logistic Regression)

In [ ]:
# Logistic Regression - Baseline Model for Classification
print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=200, random_state=42)
lr_model.fit(X_train, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test)
y_pred_proba_lr = lr_model.predict_proba(X_test)

# Evaluation
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr, average='weighted', zero_division=0)
lr_recall = recall_score(y_test, y_pred_lr, average='weighted', zero_division=0)
lr_f1 = f1_score(y_test, y_pred_lr, average='weighted', zero_division=0)

print(f"\nLogistic Regression Performance:")
print(f"  Accuracy:  {lr_accuracy:.4f}")
print(f"  Precision: {lr_precision:.4f}")
print(f"  Recall:    {lr_recall:.4f}")
print(f"  F1-Score:  {lr_f1:.4f}")

# Cross-validation scores
lr_cv_scores = cross_val_score(lr_model, X_train, y_train, cv=kfold, scoring='accuracy')
print(f"  CV Scores: {lr_cv_scores}")
print(f"  CV Mean:   {lr_cv_scores.mean():.4f} (+/- {lr_cv_scores.std():.4f})")

baseline_results = {
    'Logistic Regression': {
        'accuracy': lr_accuracy,
        'precision': lr_precision,
        'recall': lr_recall,
        'f1': lr_f1,
        'model': lr_model
    }
}

print("\n✓ Baseline model training completed!")

## 8. Tree-Based Models (Decision Tree, Random Forest)

In [ ]:
# Decision Tree Classifier
print("Training Decision Tree Classifier...")
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)
y_pred_proba_dt = dt_model.predict_proba(X_test)

dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt, average='weighted', zero_division=0)
dt_recall = recall_score(y_test, y_pred_dt, average='weighted', zero_division=0)
dt_f1 = f1_score(y_test, y_pred_dt, average='weighted', zero_division=0)

print(f"\nDecision Tree Performance:")
print(f"  Accuracy:  {dt_accuracy:.4f}")
print(f"  Precision: {dt_precision:.4f}")
print(f"  Recall:    {dt_recall:.4f}")
print(f"  F1-Score:  {dt_f1:.4f}")

# Random Forest Classifier
print("\nTraining Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf, average='weighted', zero_division=0)
rf_recall = recall_score(y_test, y_pred_rf, average='weighted', zero_division=0)
rf_f1 = f1_score(y_test, y_pred_rf, average='weighted', zero_division=0)

print(f"\nRandom Forest Performance:")
print(f"  Accuracy:  {rf_accuracy:.4f}")
print(f"  Precision: {rf_precision:.4f}")
print(f"  Recall:    {rf_recall:.4f}")
print(f"  F1-Score:  {rf_f1:.4f}")

# Store results
baseline_results['Decision Tree'] = {
    'accuracy': dt_accuracy,
    'precision': dt_precision,
    'recall': dt_recall,
    'f1': dt_f1,
    'model': dt_model
}

baseline_results['Random Forest'] = {
    'accuracy': rf_accuracy,
    'precision': rf_precision,
    'recall': rf_recall,
    'f1': rf_f1,
    'model': rf_model
}

# Feature importance
print("\nFeature Importance (Random Forest):")
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance)

print("\n✓ Tree-based models training completed!")

## 9. Hyperparameter Tuning (GridSearch)

In [ ]:
# Hyperparameter tuning for Random Forest
print("Tuning Random Forest Hyperparameters...")

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# GridSearchCV
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

print("Running GridSearchCV (this may take a moment)...")
grid_search.fit(X_train, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

# Test the best model
best_rf = grid_search.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)

best_rf_accuracy = accuracy_score(y_test, y_pred_best_rf)
best_rf_f1 = f1_score(y_test, y_pred_best_rf, average='weighted', zero_division=0)

print(f"\nBest Random Forest Test Performance:")
print(f"  Accuracy: {best_rf_accuracy:.4f}")
print(f"  F1-Score: {best_rf_f1:.4f}")

# GridSearch results
cv_results = pd.DataFrame(grid_search.cv_results_)
print("\nTop 5 parameter combinations:")
print(cv_results[['param_n_estimators', 'param_max_depth', 'param_min_samples_split', 'mean_test_score']].head())

print("\n✓ Hyperparameter tuning completed!")

## 10. Model Evaluation Metrics

In [ ]:
# Comprehensive evaluation metrics
models_to_eval = {
    'Logistic Regression': (lr_model, y_pred_lr, y_pred_proba_lr),
    'Decision Tree': (dt_model, y_pred_dt, y_pred_proba_dt),
    'Random Forest': (rf_model, y_pred_rf, y_pred_proba_rf),
    'Tuned Random Forest': (best_rf, y_pred_best_rf, best_rf.predict_proba(X_test))
}

evaluation_results = {}

for model_name, (model, y_pred, y_pred_proba) in models_to_eval.items():
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    evaluation_results[model_name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

# Create comparison table
eval_df = pd.DataFrame(evaluation_results).T
print("Model Performance Comparison:")
print(eval_df.round(4))

# Save to CSV
eval_df.to_csv('../models/model_evaluation.csv')
print("\n✓ Evaluation metrics saved to model_evaluation.csv")

## 11. Visualization: Confusion Matrix & ROC Curve

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (model_name, (model, y_pred, y_pred_proba)) in enumerate(models_to_eval.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(f'Confusion Matrix - {model_name}')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('../models/confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrices saved!")

# ROC Curves (binary classification: class 1 vs rest)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (model_name, (model, y_pred, y_pred_proba)) in enumerate(models_to_eval.items()):
    # For multiclass, we'll use the macro-averaged approach
    # Plot ROC curve for each class
    from sklearn.preprocessing import label_binarize
    y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
    
    fpr, tpr, roc_auc = {}, {}, {}
    
    for i in range(3):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    
    ax = axes[idx]
    for i in range(3):
        ax.plot(fpr[i], tpr[i], label=f'Class {i} (AUC = {roc_auc[i]:.2f})')
    
    ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve - {model_name}')
    ax.legend(loc="lower right")
    ax.grid(True)

plt.tight_layout()
plt.savefig('../models/roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ ROC curves saved!")

# Feature Importance Plot
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'])
plt.xlabel('Importance')
plt.title('Feature Importance - Random Forest')
plt.tight_layout()
plt.savefig('../models/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Feature importance plot saved!")

## 12. Model Persistence (Save/Load)

In [ ]:
# Save models and preprocessing objects
model_save_dir = Path('../models')
model_save_dir.mkdir(exist_ok=True)

# Save best model
joblib.dump(best_rf, model_save_dir / 'best_random_forest_model.pkl')
print("✓ Saved: best_random_forest_model.pkl")

# Save scaler
joblib.dump(scaler, model_save_dir / 'scaler.pkl')
print("✓ Saved: scaler.pkl")

# Save selector
joblib.dump(selector, model_save_dir / 'feature_selector.pkl')
print("✓ Saved: feature_selector.pkl")

# Create model metadata
metadata = {
    'model_name': 'Best Random Forest Classifier',
    'training_accuracy': best_rf_accuracy,
    'f1_score': best_rf_f1,
    'best_params': grid_search.best_params_,
    'feature_names': X_train.columns.tolist(),
    'classes': best_rf.classes_.tolist()
}

import json
with open(model_save_dir / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4, default=str)
print("✓ Saved: model_metadata.json")

# Demonstrate loading and using the model
print("\n--- Loading Model and Making Predictions ---")
loaded_model = joblib.load(model_save_dir / 'best_random_forest_model.pkl')
loaded_scaler = joblib.load(model_save_dir / 'scaler.pkl')

# Make predictions on new data
sample_predictions = loaded_model.predict(X_test[:5])
sample_proba = loaded_model.predict_proba(X_test[:5])

print("\nSample predictions from loaded model:")
for i in range(5):
    print(f"  Sample {i+1}: Predicted class {sample_predictions[i]}, Probabilities: {sample_proba[i].round(3)}")

print("\n✓ Model persistence and loading completed!")

## 13. Reproducibility & Experiment Logging

In [ ]:
import datetime

# Set random seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Experiment logging
experiment_log = {
    'timestamp': datetime.datetime.now().isoformat(),
    'dataset': 'Iris',
    'test_size': 0.2,
    'random_state': RANDOM_STATE,
    'preprocessing': {
        'scaler': 'StandardScaler',
        'missing_values': 'None',
        'feature_selection': 'SelectKBest (k=3)'
    },
    'models_trained': list(models_to_eval.keys()),
    'best_model': 'Tuned Random Forest',
    'hyperparameters_grid': param_grid,
    'best_parameters': grid_search.best_params_,
    'evaluation_metrics': evaluation_results
}

# Save experiment log
with open(model_save_dir / 'experiment_log.json', 'w') as f:
    json.dump(experiment_log, f, indent=4, default=str)

print("Experiment Log:")
print(json.dumps(experiment_log, indent=2, default=str))

# Create a summary report
summary_report = f"""
===============================================
MACHINE LEARNING EXPERIMENT SUMMARY REPORT
===============================================

Experiment Date: {experiment_log['timestamp']}
Dataset: {experiment_log['dataset']}
Test Size: {experiment_log['test_size']}
Random State: {experiment_log['random_state']}

PREPROCESSING:
- Scaler: {experiment_log['preprocessing']['scaler']}
- Missing Values: {experiment_log['preprocessing']['missing_values']}
- Feature Selection: {experiment_log['preprocessing']['feature_selection']}

MODELS TRAINED:
"""

for model in experiment_log['models_trained']:
    summary_report += f"  - {model}\n"

summary_report += f"""
HYPERPARAMETER TUNING:
- Grid Search CV Folds: 5
- Scoring Metric: f1_weighted
- Best Parameters: {experiment_log['best_parameters']}

BEST MODEL PERFORMANCE:
- Accuracy:  {best_rf_accuracy:.4f}
- F1-Score:  {best_rf_f1:.4f}

MODEL COMPARISON:
"""

for model_name, metrics in experiment_log['evaluation_metrics'].items():
    summary_report += f"\n  {model_name}:\n"
    for metric, value in metrics.items():
        summary_report += f"    - {metric}: {value:.4f}\n"

summary_report += """
ARTIFACTS SAVED:
- best_random_forest_model.pkl: Trained model
- scaler.pkl: Feature scaler
- feature_selector.pkl: Feature selector
- model_metadata.json: Model metadata
- experiment_log.json: Experiment log
- confusion_matrices.png: Confusion matrices visualization
- roc_curves.png: ROC curves visualization
- feature_importance.png: Feature importance plot
- model_evaluation.csv: Evaluation metrics table

===============================================
"""

print(summary_report)

# Save summary report
with open(model_save_dir / 'experiment_summary.txt', 'w') as f:
    f.write(summary_report)

print("✓ Experiment summary saved to experiment_summary.txt")
print("\n✓ All experiments logged and reproducible!")